In [1]:
import numpy as np
import joblib

import pandas as pd
pd.set_option("display.max_columns", None)


In [2]:
dados = pd.read_csv('../data/klike_challenge_dataset.csv')
dados.head(5)

,campaign_id,date,platform,category,objective,target_audience_age,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density,music_voice_ratio,impressions,clicks,ctr,cpc,spend,conversions,revenue,roas,avg_watch_time_s,engagement_rate,klike_score
0,KLK-0001,2024-01-01,Meta,Branding,awareness,18-24,False,NaN,vertical,True,True,False,True,medium,0.39,150825,5882,0.0390,0.82,4823.24,48,0.00,0.00,3.9,0.0446,63.6
1,KLK-0002,2024-03-22,LinkedIn,App Install,traffic,25-34,False,NaN,horizontal,True,True,True,True,low,0.07,7433,493,0.0664,3.54,1745.22,34,NaN,0.16,6.4,0.0217,70.0
2,KLK-0003,2024-05-04,TikTok,SaaS,traffic,25-34,False,8.0,vertical,True,True,True,True,medium,0.43,193198,15069,0.0780,0.38,5726.22,492,51053.44,8.92,8.0,0.1000,99.1
3,KLK-0004,2024-12-27,TikTok,Lead Gen,engagement,25-34,False,27.0,quadrado,False,True,True,True,medium,0.54,6506,601,0.0925,0.71,426.71,19,1113.78,2.61,20.3,0.0414,74.9
4,KLK-0005,2024-07-10,Meta,Branding,engagement,25-34,False,34.0,vertical,True,True,True,True,low,0.35,63830,2355,0.0369,0.38,894.90,13,0.00,0.00,34.0,0.0184,77.9


In [3]:
df_hook = dados.groupby(['platform', 'has_hook'], observed=True)['roas'].agg(['count', 'median'])
com_hook = df_hook.xs(True, level='has_hook')['median']
sem_hook = df_hook.xs(False, level='has_hook')['median']

diferenca_pct = ((com_hook - sem_hook) / sem_hook) * 100
display(diferenca_pct.round(2))
df_hook

platform
LinkedIn     0.00
Meta       -27.07
TikTok      35.91
Name: median, dtype: float64

count  median
platform has_hook               
LinkedIn False        38    0.53
         True         45    0.53
Meta     False       113    1.81
         True        128    1.32
TikTok   False        93    2.20
         True         83    2.99

In [4]:
# Funções ----------------------------------------------------------------------------------------------------------------------

def comparar_atributo(dados, atributo, metrica, atual, alternativa):
    df = dados.groupby(['platform', atributo], observed=True)[metrica].agg(['count', 'median'])
    grupo_atual = df.xs(atual, level=atributo)['median']
    grupo_alternativa = df.xs(alternativa, level=atributo)['median']
    diferenca_pct = ((grupo_alternativa - grupo_atual) / grupo_atual) * 100
    return diferenca_pct

def melhor_alternativa(dados, plataforma, atributo, metrica, valor_atual):
    df = dados[dados['platform'] == plataforma]
    resumo = df.groupby(atributo, observed=True)[metrica].agg(['count', 'median'])
    # Considera só grupos com pelo menos 10 campanhas
    resumo = resumo[resumo['count'] >= 10]

    if valor_atual not in resumo.index:
        return None

    mediana_atual = resumo.loc[valor_atual, 'median']
    # Remove a categoria atual das alternativas
    alternativas = resumo.drop(index=valor_atual)

    if alternativas.empty or mediana_atual <= 0:
        return None

    # Seleciona alternativa com maior mediana
    melhor_valor = alternativas['median'].idxmax()
    melhor_mediana = alternativas.loc[melhor_valor, 'median']

    diferenca_pct = (melhor_mediana - mediana_atual) / mediana_atual * 100

    if isinstance(valor_atual, bool) and isinstance(melhor_valor, bool):
        valor_atual = bool(valor_atual)
        melhor_valor = bool(melhor_valor)

    if diferenca_pct <= 0:
        return None

    return {'atributo': atributo, 'valor_atual': valor_atual, 'valor_recomendado': melhor_valor, 
            'metrica': metrica, 'diferenca_pct': float(diferenca_pct), 'mediana_atual': float(mediana_atual),
            'mediana_alternativa': float(melhor_mediana), 'n_atual': int(resumo.loc[valor_atual, 'count']),
            'n_alternativa': int(resumo.loc[melhor_valor, 'count'])}

def metrica_por_objetivo(objetivo):
    metricas_por_objetivo = {
    "traffic": "ctr",
    "engagement": "engagement_rate",
    "conversions": "roas",
    "awareness": "watch_time_ratio",
    "app_install": "conversion_rate"}
    return metricas_por_objetivo[objetivo]

def gerar_candidatas(campanha, dados):
    candidatas = []
    # Identificar a métrica principal da campanha
    metrica_principal = metrica_por_objetivo(campanha['objective'])
    metricas = ['ctr', 'conversion_rate', 'roas', 'watch_time_ratio', 'engagement_rate']
    metricas_ordenadas = [metrica_principal] + [m for m in metricas if m != metrica_principal]

    # Percorrer métricas e atributos modificáveis
    atributos = ["has_hook", "has_cta", "has_face", "has_subtitle", "text_density", "duration_group", "audio_group", "format"]
    for atributo in atributos:
        if pd.isna(campanha[atributo]):
            continue
        for metrica in metricas_ordenadas:
            resultado = melhor_alternativa(dados=dados, plataforma=campanha['platform'], atributo=atributo, metrica=metrica, 
                                                         valor_atual=campanha[atributo])
            if resultado is not None:
                resultado["metrica_principal"] = (metrica == metrica_principal)
                candidatas.append(resultado)
    
    return candidatas

def priorizar_recomendacoes(candidatas, max_recs=5, diferenca_minima=10):
    recs_finais = []
    
    recs_metrica_principal = [rec for rec in candidatas if rec["metrica_principal"]]
    recs_metrica_principal = sorted(recs_metrica_principal, key=lambda rec: rec["diferenca_pct"], reverse=True)
    
    recs_outras = [rec for rec in candidatas if not rec["metrica_principal"]]
    recs_outras = sorted(recs_outras, key=lambda rec: rec["diferenca_pct"], reverse=True)

    atributos_selecionados = set()

    for rec in recs_metrica_principal:
        if len(recs_finais) >= min(2, max_recs):
            break

        if rec["diferenca_pct"] >= diferenca_minima and rec["atributo"] not in atributos_selecionados:
            recs_finais.append(rec)
            atributos_selecionados.add(rec["atributo"])

    for rec in recs_outras:
        if len(recs_finais) >= max_recs:
            break

        if rec["diferenca_pct"] >= diferenca_minima and rec["atributo"] not in atributos_selecionados:
            recs_finais.append(rec)
            atributos_selecionados.add(rec["atributo"])

    recs_finais = sorted(recs_finais, key=lambda rec: rec["diferenca_pct"], reverse=True)
    return recs_finais

def aplicar_recomendacoes(campanha, recomendacoes, dados):
    campanha_modificada = campanha.copy()
    for rec in recomendacoes:
        atributo = rec["atributo"]

        if atributo in ["duration_group", "audio_group"]:
            coluna_original = {"duration_group": "video_duration_s", "audio_group": "music_voice_ratio"}[atributo]
            valor_recomendado = dados.loc[(dados["platform"] == campanha["platform"]) & (dados[atributo] == rec["valor_recomendado"]), 
                                          coluna_original].median()
            campanha_modificada[coluna_original] = float(valor_recomendado)
        else:
            campanha_modificada[atributo] = rec["valor_recomendado"]
            
    return campanha_modificada

def avaliar_score_candidatas(camapanha, candidatas, dados, preprocessor, ridge, features):
    candidatas_avaliadas = []

    X_original = camapanha.to_frame().T[features]
    score_original = ridge.predict(preprocessor.transform(X_original))[0]

    for rec in candidatas:
        campanha_modificada = aplicar_recomendacoes(camapanha, [rec], dados)
        X_modificada = campanha_modificada.to_frame().T[features]
        
        score_modificado = ridge.predict(preprocessor.transform(X_modificada))[0]

        rec_avaliada = rec.copy()
        rec_avaliada["delta_score"] = float(score_modificado - score_original)

        candidatas_avaliadas.append(rec_avaliada)

    return candidatas_avaliadas

In [5]:
# Criando grupos de duração para tratar como categórica

dados['duration_group'] = pd.qcut(dados['video_duration_s'], q=3, labels=['short', 'medium', 'long'])

df_duration = dados.groupby(['platform', 'duration_group'], observed=True)['engagement_rate'].agg(['count', 'median'])
display(df_duration)

count   median
platform duration_group                
LinkedIn short               7  0.01250
         medium             20  0.01620
         long               44  0.01820
Meta     short              71  0.01830
         medium             65  0.01660
         long               72  0.01560
TikTok   short              73  0.03750
         medium             56  0.03535
         long               24  0.02245

In [6]:
# Criando grupos de audio para tratar como categórica

dados['audio_group'] = pd.qcut(dados['music_voice_ratio'], q=3, labels=['low_music', 'medium_music', 'high_music'])

df_audio = dados.groupby(['platform', 'audio_group'], observed=True)['engagement_rate'].agg(['count', 'median'])
display(df_audio)

dados['music_voice_ratio'].quantile([0, 1/3, 2/3, 1])

count   median
platform audio_group                 
LinkedIn low_music        27  0.01840
         medium_music     23  0.01230
         high_music       19  0.02280
Meta     low_music        73  0.01640
         medium_music     65  0.02040
         high_music       74  0.01750
TikTok   low_music        51  0.02820
         medium_music     54  0.03005
         high_music       50  0.03755

0.000000    0.02
0.333333    0.29
0.666667    0.49
1.000000    0.92
Name: music_voice_ratio, dtype: float64

In [7]:
# Criando taxa de conversão com base na quantidade de cliques

dados["conversion_rate"] = (dados["conversions"] / dados["clicks"].replace(0, np.nan))

display(dados["conversion_rate"].describe())
print("Valores ausentes:", dados["conversion_rate"].isna().sum())
print("Taxas acima de 100%:", (dados["conversion_rate"] > 1).sum())

count    500.000000
mean       0.034401
std        0.026457
min        0.000000
25%        0.015444
50%        0.030769
75%        0.046921
max        0.156250
Name: conversion_rate, dtype: float64

Valores ausentes: 0
Taxas acima de 100%: 0


In [8]:
print(dados[["avg_watch_time_s", "video_duration_s"]].isna().sum())

print("Durações iguais a zero:",
      (dados["video_duration_s"] == 0).sum())

avg_watch_time_s    27
video_duration_s    39
dtype: int64
Durações iguais a zero: 0


In [9]:
# Criando a proporção do vídeo que foi assistida

dados["watch_time_ratio"] = dados["avg_watch_time_s"] / dados["video_duration_s"].replace(0, np.nan)

display(dados["watch_time_ratio"].describe())
print("Valores ausentes:", dados["watch_time_ratio"].isna().sum())
print("Proporções acima de 100%:", (dados["watch_time_ratio"] > 1).sum())

count    437.000000
mean       0.542130
std        0.245784
min        0.033333
25%        0.361538
50%        0.508333
75%        0.728571
max        1.000000
Name: watch_time_ratio, dtype: float64

Valores ausentes: 63
Proporções acima de 100%: 0


In [10]:
# Calcular o novo klike_score
ferramentas = joblib.load("../models/ridge_model.joblib")

preprocessor = ferramentas["preprocessor"]
ridge = ferramentas["model"]

In [11]:
# CÉLULA DE TESTE --------------------------------------------------------------------------------------------
campanha = dados.iloc[4]

candidatas = gerar_candidatas(campanha, dados)
recomendacoes = priorizar_recomendacoes(candidatas)

# print(f"Total de recomendações: {len(recomendacoes)}")
# display(pd.DataFrame(recomendacoes))

campanha_modificada = aplicar_recomendacoes(campanha, recomendacoes, dados)
# comparacao = pd.DataFrame({"Original": campanha, "Modificada": campanha_modificada})
# display(comparacao)

# Visualizar
display(pd.DataFrame(recomendacoes)[["atributo", "valor_atual", "valor_recomendado", "metrica", "diferenca_pct", "metrica_principal"]])

,atributo,valor_atual,valor_recomendado,metrica,diferenca_pct,metrica_principal
0,audio_group,medium_music,high_music,ctr,42.395437,False
1,has_hook,True,False,roas,37.121212,False
2,format,vertical,quadrado,roas,30.985915,False
3,text_density,low,medium,engagement_rate,23.030303,True
4,duration_group,long,short,engagement_rate,17.307692,True


In [12]:
def recommendations_engine(campanha, dados):
    candidatas = gerar_candidatas(campanha, dados)
    recomendacoes = priorizar_recomendacoes(candidatas)

    campanha_modificada = aplicar_recomendacoes(campanha, recomendacoes, dados)

    return {"recomendacoes": recomendacoes,"campanha_modificada": campanha_modificada}

In [23]:
def descrever_acao(recomendacao, campanha, campanha_modificada):
    atributo = recomendacao["atributo"]

    acoes_booleanas = {
        "has_hook": ("Adicionar um hook nos primeiros 3 segundos",
                     "Remover o hook inicial"),
        "has_cta": ("Adicionar uma chamada para ação (CTA)",
                    "Remover a chamada para ação (CTA)"),
        "has_subtitle": ("Adicionar legendas ao vídeo",
                         "Remover as legendas do vídeo"),
        "has_face": ("Incluir um rosto humano no vídeo",
                     "Remover a presença de rosto humano")}

    if atributo in acoes_booleanas:
        novo_valor = bool(campanha_modificada[atributo])
        return acoes_booleanas[atributo][0 if novo_valor else 1]
    elif atributo == "format":
        atual = campanha["format"]
        novo = campanha_modificada["format"]
        return f"Alterar o formato de vídeo de {atual} para {novo}"
    elif atributo == "text_density":
        traducoes = {"low": "baixa", "medium": "média", "high": "alta"}
        atual = traducoes.get(campanha["text_density"], campanha["text_density"])
        novo = traducoes.get(campanha_modificada['text_density'], campanha_modificada["text_density"])
        return f"Alterar a densidade de texto de {atual} para {novo}"
    elif atributo == "duration_group":
        atual = campanha["video_duration_s"]
        novo = campanha_modificada["video_duration_s"]
        verbo = "Reduzir" if novo < atual else "Aumentar"
        return (f"{verbo} a duração do vídeo de " f"{atual:.0f}s para aproximadamente {novo:.0f}s")
    elif atributo == "audio_group":
        atual = campanha["music_voice_ratio"] * 100
        novo = campanha_modificada["music_voice_ratio"] * 100
        verbo = "Reduzir" if novo < atual else "Aumentar"
        return (f"{verbo} a proporção de música no áudio " f"de {atual:.0f}% para aproximadamente {novo:.0f}%")
    else:
        raise ValueError(f"Atributo não reconhecido: {atributo}")

def descrever_evidencia(recomendacao, plataforma):
    nomes_metricas = {
    "ctr": "CTR",
    "roas": "ROAS",
    "engagement_rate": "taxa de engajamento",
    "conversion_rate": "taxa de conversão",
    "watch_time_ratio": "proporção assistida do vídeo"}
    metrica = recomendacao["metrica"]
    diferenca = recomendacao["diferenca_pct"]
    nome_metrica = nomes_metricas[metrica]

    return (f"Campanhas com essa característica na plataforma " f"{plataforma} apresentaram {nome_metrica} mediano " 
            f"{diferenca:.1f}% superior ao grupo atual.")


def apresentar_recomendacoes(campanha, resultado):
    recomendacoes = resultado["recomendacoes"]
    campanha_modificada = resultado["campanha_modificada"]

    campanha_id = campanha["campaign_id"]
    plataforma = campanha["platform"]

    objetivos = {
    "awareness": "Reconhecimento de marca",
    "traffic": "Tráfego",
    "conversions": "Conversões",
    "engagement": "Engajamento",
    "app_install": "Instalação de aplicativo"}

    objetivo = objetivos.get(campanha["objective"], campanha["objective"])

    print("=" * 65)
    print(f"RECOMENDAÇÕES PARA A CAMPANHA {campanha_id}")
    print(f"Plataforma: {plataforma} | Objetivo: {objetivo}")
    print("=" * 65)

    if not recomendacoes:
        print("\nNenhuma recomendação válida foi identificada.")
        return
    
    print(f"\n{len(recomendacoes)} recomendações identificadas:\n")

    for i, recomendacao in enumerate(recomendacoes, start=1):
        acao = descrever_acao(recomendacao, campanha, campanha_modificada)
        evidencia = descrever_evidencia(recomendacao, plataforma)
        print(f"{i}. {acao}.")
        print(f"   {evidencia}\n")

    print("-" * 65)
    print("Nota: as recomendações são baseadas em associações observadas no dataset e não garantem melhorias de performance.")


In [24]:
campanha = dados.iloc[89]
resultado = recommendations_engine(campanha, dados)
apresentar_recomendacoes(campanha, resultado)

RECOMENDAÇÕES PARA A CAMPANHA KLK-0090
Plataforma: Meta | Objetivo: Conversões

5 recomendações identificadas:

1. Adicionar um hook nos primeiros 3 segundos.
   Campanhas com essa característica na plataforma Meta apresentaram CTR mediano 57.5% superior ao grupo atual.

2. Incluir um rosto humano no vídeo.
   Campanhas com essa característica na plataforma Meta apresentaram taxa de engajamento mediano 54.1% superior ao grupo atual.

3. Reduzir a proporção de música no áudio de 60% para aproximadamente 39%.
   Campanhas com essa característica na plataforma Meta apresentaram ROAS mediano 47.1% superior ao grupo atual.

4. Reduzir a duração do vídeo de 15s para aproximadamente 10s.
   Campanhas com essa característica na plataforma Meta apresentaram ROAS mediano 37.0% superior ao grupo atual.

5. Alterar o formato de vídeo de vertical para quadrado.
   Campanhas com essa característica na plataforma Meta apresentaram taxa de conversão mediano 26.1% superior ao grupo atual.

------------